# Denoising

# Import

In [ ]:
%pip install ipykernel kagglehub openpyxl imbalanced-learn seaborn torch plotly nbformat >= "4.2.0"

In [ ]:
from typing import Literal, Callable
from pathlib import Path
from matplotlib import pyplot as plt
from collections import Counter

import kagglehub as kg
import numpy as np
import pandas as pd
import seaborn as sns
import random

from sklearn.model_selection import StratifiedShuffleSplit, cross_val_predict
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier as MLP
from sklearn.base import ClassifierMixin
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.base import TransformerMixin
from sklearn.manifold import Isomap

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from helper.plot import plot_attributes
from helper.transform import (
    transform_pca,
    remove_outliers_zscore,
    remove_outliers_db,
    remove_outliers_isf,
    bin_attributes_mean,
    bin_attributes_median,
    regression_reduce_noise,
)
from helper.transform import (
    remove_label_noise_ensemble_filter,
    remove_label_noise_cross_validated_committees_filter,
    remove_label_noise_iterative_partitioning_filter,
)


from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay
)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Compute classification metrics for multi-class classification.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    return {
        "precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "f1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "confusion_matrix": cm,
        "n_classes": len(np.unique(y_true))
    }

# Load data

In [ ]:
path = kg.dataset_download("muratkokludataset/dry-bean-dataset")
file_name = "/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx"
print("Downloaded at: ", path)
data = pd.read_excel(path + file_name)
data

In [ ]:
def encode_n_split(
    data: pd.DataFrame,
    train_percentage: float,
    encoder: TransformerMixin,
    class_column: str,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    returns tuple containing train set, test set and string representing the label column and the trained encoder
    """
    datacp = data.copy()
    encoded_class_column = f"Class_{encoder.__class__.__name__}"
    datacp[encoded_class_column] = encoder.fit_transform(datacp[class_column])
    datacp.drop(inplace=True, axis=1, labels=[class_column])

    test_percentage = 1 - train_percentage

    split = StratifiedShuffleSplit(
        n_splits=1, test_size=test_percentage, random_state=42
    )

    # Perform the split
    for train_idx, test_idx in split.split(datacp, datacp[encoded_class_column]):
        train_set = datacp.iloc[train_idx]
        test_set = datacp.iloc[test_idx]

    print("Train size: ", len(train_set), "x", len(train_set.iloc[0]))
    print("Test size: ", len(test_set), "x", len(test_set.iloc[0]))
    print(f'Encoded classes in column "{encoded_class_column}"')
    return (train_set, test_set, encoder, encoded_class_column)

In [ ]:
train_set, test_set, encoder, encoded_class_column = encode_n_split(
    data, 0.8, LabelEncoder(), "Class"
)

In [ ]:
y_train = train_set[encoded_class_column]
y_test = test_set[encoded_class_column]
X_train = train_set.drop(axis=1, labels=[encoded_class_column])
X_test = test_set.drop(axis=1, labels=[encoded_class_column])

# EDA

In [ ]:
n_classes, class_counts = np.unique(y_train, return_counts=True)

In [ ]:
plot_attributes(X_train, label_column=encoded_class_column, n_attrs=len(class_counts))

Atributi 
- Area
- Perimeter
- Major Axis Length
- Minor Axis Length
- AspectRation
- ConvexArea
- EquivDiameter 

imaju dosta velike vrednosti dok ostali atributi su u range $[0-1]$ \
Sa grafikona vidimo da vrednosti atributa ne prate normalnu raspodelu.\
Atributi kao ShapeFactor4 i Solidity imaju velike repove. Zato ćemo tokom preprocesiranja normalizovati raspodele.

In [ ]:
_ = sns.countplot(data=y_train.to_frame(), x=encoded_class_column)

Pored toga skup podataka nije balansiran:
- Klasa 1 je slabo zastupljena sa manje od 500 instanci, klasa 0 ima oko 1000 dok klasa 3 dominira sa oko 2500 instanci
- Zato ćemo izvršiti under i oversampling na oko 500-1000 instanci

In [ ]:
sns.pairplot(pd.concat([X_train, y_train.rename("Class")], axis=1), hue="Class")
plt.show()

Po grafikonu iznad vidimo da se klasa 1 karakteriše visokim vrednostima Major i Minor axis length, EquivDiameter, ConvexArea, Area i Perimeter. Pored toga ima male vrednosti ShapeFactor1.

<div style="background-color:#1212AA; height:auto; border-radius:10px; padding:16px; width:600px; color:white">
<h3>Zaključci</h3>
<ul>
<li>Podaci su nebalansirani, potrebno je under i over sample-ovati na oko 1-1.5 hiljade instanci</li>
<li>Atribute je potrebno normalizovati, a neke i skalirati kao što su Area</li>
<li>Klasa 1 je nedovoljno zastupljena ali lako prepoznatljiva po atributima koji imaju visoke vrednosti kao što je Area</li>
</ul>
</div>


# Normalizacija/Standardizacija

In [ ]:
pt = PowerTransformer(method="yeo-johnson")
X_train_transformed = pt.fit_transform(X_train)

In [ ]:
# Samo power transform
X_train_transformed = pd.DataFrame(X_train_transformed, columns=X_train.columns)
sns.pairplot(
    pd.concat([X_train_transformed, y_train.rename("Class")], axis=1), hue="Class"
)
plt.show()

In [ ]:
# PowerTransform + MinMaxScaler
scaler = MinMaxScaler()
train_set_scaled = scaler.fit_transform(X_train_transformed)
X_train_transformed = pd.DataFrame(train_set_scaled, columns=X_train.columns)
sns.pairplot(
    pd.concat([X_train_transformed, y_train.rename("Class")], axis=1), hue="Class"
)
plt.show()

In [ ]:
# Samo MinMaxScaler
scaler = MinMaxScaler()
train_set_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(train_set_scaled, columns=X_train.columns)
sns.pairplot(pd.concat([X_train_scaled, y_train.rename("Class")], axis=1), hue="Class")
plt.show()

## TestClassifiers (refactored to return structured results)

In [ ]:

def TestClassifiers(
    dataPredictorPairs,
    noise_instances = None,
    noise_amp = None,
    noise_type = "attributes",
    method="baseline",
    n_columns=3,
    normalize="pred"
):
    results = []

    n = len(dataPredictorPairs)
    if n < n_columns:
        n_columns = n
    n_rows = (n + n_columns - 1) // n_columns

    fig, axes = plt.subplots(
        n_rows, n_columns,
        figsize=(5*n_columns, 4*n_rows),
        squeeze=False
    )

    for idx, (X, y, clf, difficulty, name, X_test, y_test) in enumerate(dataPredictorPairs):
        r, c = divmod(idx, n_columns)
        ax = axes[r][c]
        
        if X_test is None:
            # Use cross-validation on X
            from sklearn.model_selection import cross_val_predict
            y_pred = cross_val_predict(clf, X, y, cv=3)
            metrics = compute_metrics(y, y_pred)
        else:
            # Train on X, predict on X_test
            clf.fit(X, y)
            y_pred = clf.predict(X_test)
            metrics = compute_metrics(y_test, y_pred)
            y = y_test  # Use test labels for confusion matrix
        
        classifier_name = clf.__class__.__name__
        
        results.append({
            "method": method,
            "noise_level": noise_instances if noise_instances is not None else "NA",
            "noise_amp": noise_amp if noise_amp is not None else "NA",
            "noise_type": noise_type,
            "difficulty": difficulty if difficulty is not None else "NA",
            "test_case": name,
            "classifier": classifier_name,
            **metrics
        })
        
        ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=ax, normalize=normalize)
        ax.set_title(f"{name}\nF1={metrics['f1']:.3f}")

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(results)


Na osnovu prethodnih grafikona mozemo da zakljucimo da `PowerTransform` i `MinMaxScaler` pogorsavaju separaciju razlicitih klasa.\

In [ ]:
TestClassifiers(
    dataPredictorPairs=[
        (
            X_train,
            y_train,
            RandomForestClassifier(n_jobs=-1, n_estimators=20),
            0,
            "Without transforms",
            None,
            None,
        ),
        (
            X_train_transformed,
            y_train,
            RandomForestClassifier(n_jobs=-1, n_estimators=20),
            0,
            "With power and scaling transforms",
            None,
            None,
        ),
        (
            X_train_scaled,
            y_train,
            RandomForestClassifier(n_jobs=-1, n_estimators=20),
            0,
            "With scaling only",
            None,
            None,
        ),
    ]
)

In [ ]:
TestClassifiers(
    dataPredictorPairs=[
        (X_train, y_train, SVC(), 0, "Without transforms", None, None),
        (X_train_transformed, y_train, SVC(), 0, "With power and scaling transforms", None, None),
        (X_train_scaled, y_train, SVC(), 0, "With scaling only", None, None),
    ]
)

Međutim, rezultati pokazuju bolji accuracy kada se primeni Power i Scaling transformacija. Ovo se vidi i po accuracy score-u kao i po konfuzionoj matrici.\
Zato na dalje koristimo transformisani skup podataka.\
Iako ima malo instanci klase 1, klasifikacija ovih instanci je jednostavna za oba modela.
Sa druge strane, iako model ima najviše instanci klase 3 ova klasa je najteža za klasifikovati. Ovo ukazuje na overfitting ili šum među instancama klase 3.

# Under/OverSampling

In [ ]:
target_samples = 1700  # Pick a number in the range

# Oversample with SMOTE and undersample with RandomUnderSampler
smote = SMOTE(
    sampling_strategy=lambda y: {
        k: max(target_samples, v) for k, v in Counter(y).items() if v < target_samples
    },
    random_state=42,
)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

under = RandomUnderSampler(
    sampling_strategy=lambda y: {
        k: min(target_samples, v) for k, v in Counter(y).items() if v > target_samples
    },
    random_state=42,
)
X_train_resampled, y_train_resampled = under.fit_resample(
    X_train_resampled, y_train_resampled
)

In [ ]:
_ = sns.countplot(data=y_train_resampled.to_frame(), x=encoded_class_column)

In [ ]:
TestClassifiers(
    dataPredictorPairs=[
        (
            X_train_transformed,
            y_train,
            RandomForestClassifier(n_jobs=-1, n_estimators=20),
            0,
            "RandomForest, No Over/Undersample",
            None, None,
        ),
        (
            X_train_resampled,
            y_train_resampled,
            RandomForestClassifier(n_jobs=-1, n_estimators=20),
            0,
            "RandomForest + SMOTE + Undersample",
            None, None,
        ),
        (X_train_transformed, y_train, SVC(), 0, "SVC, No Over/Undersample", None, None),
        (X_train_resampled, y_train_resampled, SVC(), 0, "SVC + SMOTE + Undersample", None, None),
    ],
    n_columns=4,
)

Vršenjem under i oversamplinga dobijamo malo bolje rezultate, pa na dalje koristimo `X_train_resampled`

# Difficulty split

In [ ]:
def difficulty_split(X, y, models, difficulty_categories=3, cv=5):
    """
    Estimate difficulty of each sample based on how many models misclassify it during cross-validation.
    Returns both positional indices (for array indexing) and label indices (for .loc).
    """
    wrong_prediction_count = np.zeros(len(y))

    for model in models:
        y_pred = cross_val_predict(model, X, y, cv=cv)
        wrong_prediction_count += (y_pred != y).astype(int)

    # Define bins for difficulty categories (e.g. 0 wrong -> easy, 1-2 -> medium, etc.)
    bins = np.linspace(0, len(models), num=difficulty_categories + 1)[
        1:-1
    ]  # exclude first and last bin edges
    difficulty_labels = np.digitize(wrong_prediction_count, bins, right=False)

    # Map difficulty labels to positional indices (for array/plot indexing)
    difficulty_indices_pos = {
        int(i): np.where(difficulty_labels == i)[0].tolist()
        for i in np.unique(difficulty_labels)
    }

    # Map difficulty labels to label-based indices (for .loc)
    difficulty_indices_loc = {
        int(i): X.index[np.where(difficulty_labels == i)[0]].tolist()
        for i in np.unique(difficulty_labels)
    }

    return wrong_prediction_count, difficulty_indices_pos, difficulty_indices_loc

In [ ]:
rfc = RandomForestClassifier(n_jobs=-1)
knc = KNeighborsClassifier(n_jobs=-1)
gbc = GradientBoostingClassifier()
svc = SVC()
mlp = MLP(hidden_layer_sizes=[16, 16, 16], alpha=0)
miss_count, indices_pos, indices = difficulty_split(
    X_train_resampled,
    y_train_resampled,
    models=[rfc, knc, gbc, svc, mlp],
    difficulty_categories=3,
)

In [ ]:
def plot_difficulties_here(X, indices_map):
    iso = Isomap(n_components=2, n_jobs=-1)
    X_lowered = iso.fit_transform(X)

    color_map = {1: "green", 2: "orange", 3: "red"}

    plt.figure(figsize=(5, 5))

    for difficulty, idxs in indices_map.items():
        color = color_map.get(
            difficulty, "blue"
        )  # Default color if difficulty is missing
        sns.scatterplot(
            x=X_lowered[idxs, 0],
            y=X_lowered[idxs, 1],
            color=color,
            alpha=0.5,
            edgecolor="k",
            label=f"Difficulty {difficulty}",
        )

    plt.legend()
    plt.xlabel("Isomap Dimension 1")
    plt.ylabel("Isomap Dimension 2")
    plt.title("Instance Difficulty Visualization")
    plt.show()

In [ ]:
plot_difficulties_here(X_train_resampled, indices_pos)
for k in indices_pos.keys():
    print(f"{len(indices_pos[k])} instances in difficulty {k}.")

# Baseline performance

In [ ]:
def generate_test_pairs(
    X: pd.DataFrame,
    y: pd.DataFrame,
    models: list[ClassifierMixin],
    method: str,
    difficulty_indices: dict[int, list[int]],
    X_test: pd.DataFrame = None,
    y_test: pd.DataFrame = None,
) -> list[tuple]:
    results = []

    for model in models:
        for difficulty, idx in difficulty_indices.items():
            X_train = X.reindex(idx).dropna()
            y_train = y.reindex(idx).dropna()

            X_test_proc, y_test_proc = None, None
            if X_test and y_test:
                X_test_proc = X_test.reindex(idx).dropna()
                y_test_proc = y_test.reindex(idx).dropna()

            results.append(
                (
                    X_train,
                    y_train,
                    model,
                    difficulty,
                    f"{model.__class__.__name__}_{method}_{difficulty}",
                    X_test_proc,
                    y_test_proc,
                )
            )
    return results

# Dodavanje suma

In [ ]:
def get_noise_scheduled_indices(
    difficulty_indices: dict[int, list[int]],
    difficulty_noise_schedule: dict[int, float] | None,
) -> tuple[list[int], dict[int, list[int]]] | None:
    indices = []
    difficulty_map = {}
    for difficulty, schedule in difficulty_noise_schedule.items():
        if difficulty not in difficulty_indices:
            print(f"difficulty {difficulty} not present in the difficulty indices map")
            continue
        selected_indices = difficulty_indices[difficulty]
        random.shuffle(selected_indices)
        count = int(len(selected_indices) * schedule)
        selected = selected_indices[:count]
        difficulty_map[difficulty] = selected
        indices.extend(selected)
    return indices, difficulty_map


def get_static_schedule_instances(
    X: pd.DataFrame, instance_percentage: float
) -> list[int]:
    indices = list(X.index)
    random.shuffle(indices)
    count = int(len(indices) * instance_percentage)
    return indices[:count]


def get_attributes(
    X: pd.DataFrame, attributes: list[str] | int | Callable[[int], int]
) -> list[str]:
    attr_columns = X.columns.tolist()
    static_attrs: list[str] = []
    if isinstance(attributes, list):
        static_attrs = attributes
    elif isinstance(attributes, int):
        static_attrs = random.sample(attr_columns, min(attributes, len(attr_columns)))
    elif callable(attributes):
        count = attributes(len(attr_columns))
        static_attrs = random.sample(attr_columns, min(count, len(attr_columns)))

    return static_attrs


def generate_noise(
    X: pd.DataFrame,
    Y: pd.DataFrame,
    instance_percentage: float,
    noise_amp: float,
    attributes: list[str] | int | Callable[[int], int],
    replace: bool = True,
    regenerate_attributes: bool = False,
    apply_to: Literal["both", "attributes", "classes"] = "attributes",
    class_noise_map: dict[int, int] | None = None,
    swap_with_same_class_allowed: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, list[int], list[str]]:
    """
    Generates a noisy version of the input dataset.

    X, Y - attributes and labels
    instance_percentage - which percentage of instances are affected by noise
    noise_amp - amplitude of noise applied
    attributes - list of attributes / number of attributes / function based on which to pick attributes
    replace - do we replace or add new instances
    regenerate_attributes -  do we pick random attributes every time
    apply_to - do we apply to attributes, classes or both
    class_noise_map - how many instances to apply noise to for each class
    swap_with_same_class_allowed - can we replace class with same class or only different classes
    """

    X_result = X.copy()
    Y_result = Y.copy()

    if len(X_result) != len(Y_result):
        print(
            f"Input attributes and classes must have the same length!, X: {len(X_result)}, Y: {len(Y_result)}"
        )
        return None

    instance_indices = get_static_schedule_instances(X_result, instance_percentage)

    noisy_instances = X_result.loc[instance_indices].copy()
    noisy_classes = Y_result.loc[instance_indices].copy()

    # 2. Determine attribute columns
    attrs = get_attributes(X_result, attributes)

    # 3. Apply noise to attributes
    eps = 10e-3
    if apply_to in ["both", "attributes"]:
        for idx in instance_indices:
            if regenerate_attributes:
                attrs = get_attributes(X_result, attributes)

            for attr in attrs:
                min_val = X_result[attr].min()
                max_val = X_result[attr].max()
                current_val = noisy_instances.loc[idx, attr]
                noise = random.uniform(-1, 1)
                if abs(noise) < eps:
                    noise = -eps if noise < 0 else eps
                noise *= noise_amp
                noisy_val = current_val + noise
                noisy_instances.loc[idx, attr] = max(min_val, min(max_val, noisy_val))

    # 4. Apply noise to class labels
    if apply_to in ["both", "classes"]:
        class_values = Y_result.unique().tolist()

        for idx in instance_indices:
            current_label = noisy_classes.loc[idx]

            if class_noise_map is not None and current_label in class_noise_map:
                new_label = class_noise_map[current_label]
            else:
                if swap_with_same_class_allowed:
                    new_label = random.choice(class_values)
                else:
                    # Pick a different class
                    other_classes = [c for c in class_values if c != current_label]
                    if other_classes:
                        new_label = random.choice(other_classes)
                    else:
                        new_label = current_label  # fallback if only one class

            noisy_classes.loc[idx] = new_label

    # 5. Replace or append
    if replace:
        X_result.loc[instance_indices] = noisy_instances.values
        Y_result.loc[instance_indices] = noisy_classes.values
        instance_indices_out = instance_indices
    else:
        noisy_instances_reset = noisy_instances.reset_index(drop=True)
        noisy_classes_reset = noisy_classes.reset_index(drop=True)
        X_result = pd.concat([X_result, noisy_instances_reset], ignore_index=True)
        Y_result = pd.concat([Y_result, noisy_classes_reset], ignore_index=True)
        instance_indices_out = list(
            range(len(X_result) - len(noisy_instances_reset), len(X_result))
        )

    return X_result, Y_result, instance_indices_out

# Testiranje metoda uklanjanja šuma

In [ ]:

def evaluate_cleaning(true_noisy_idx, removed_idx, n):
    true_noisy_idx = set(true_noisy_idx)
    removed_idx = set(removed_idx)

    tp = len(true_noisy_idx & removed_idx)
    fp = len(removed_idx - true_noisy_idx)
    fn = len(true_noisy_idx - removed_idx)
    tn = n - tp - fp - fn

    return {
        "clean_tp": tp,
        "clean_fp": fp,
        "clean_fn": fn,
        "clean_tn": tn,
        "clean_precision": tp / (tp + fp + 1e-9),
        "clean_recall": tp / (tp + fn + 1e-9),
        "clean_f1": 2 * tp / (2 * tp + fp + fn + 1e-9)
    }


In [ ]:

def run_experiment(X: pd.DataFrame, 
                   y: pd.DataFrame, 
                   noise_levels: list[float], 
                   noise_amps: list[float], 
                   apply_to: list[str], 
                   models: list[ClassifierMixin],
                   difficulties: dict[int, list[int]],
                   method_name: str, 
                   transform_fn=None):
    dfs = []
    n_attrs = len(X.attrs) ** 0.5

    for apply in apply_to:
        for noise in noise_levels:
            for noise_amp in noise_amps:
                
                X_noisy, y_noisy, noisy_idx = X, y, None
                if noise is not None and noise_amp is not None:
                    X_noisy, y_noisy, noisy_idx = generate_noise(
                        X, 
                        y, 
                        instance_percentage = noise,
                        attributes=n_attrs,
                        noise_amp = noise_amp, 
                        apply_to = apply)
                
                removed_idx = []

                X_clean, y_clean = X_noisy, y_noisy
                X_test, y_test = None, None
                if transform_fn:
                    res = transform_fn(X_noisy, y_noisy)
                    if len(res) == 3:
                        X_clean, y_clean, removed_idx = res
                        X_test, y_test = X, y
                    else:
                        X_clean, y_clean = res

                name = f"{method_name}_{apply}_{noise}x{noise_amp}"
                if noise is None:
                    name = f"{method_name}_{apply}_NoNoise"

                pairs = generate_test_pairs(X_clean, y_clean, models, name, difficulties, X_test, y_test)

                df = TestClassifiers(pairs, noise, noise_amp, apply, method = method_name)

                if noisy_idx and removed_idx:
                    clean_metrics = evaluate_cleaning(noisy_idx, removed_idx, len(y))
                    for k, v in clean_metrics.items():
                        df[k] = v

                dfs.append(df)
                
                if noise is None:
                    break

    return pd.concat(dfs, ignore_index=True)


In [ ]:

def compare_to(results, base_method):
    base = results[results.method == base_method]
    comp = results.merge(
        base,
        on=["noise_level","noise_amp","noise_type", "difficulty", "classifier"],
        suffixes=("", "_base")
    )
    comp["f1_delta"] = comp["f1"] - comp["f1_base"]
    comp["precision_delta"] = comp["precision"] - comp["precision_base"]
    return comp


In [ ]:
import warnings
warnings.filterwarnings("ignore")
# =========================
# RUN ALL EXPERIMENTS
# =========================

NOISE_LEVELS = [None, 0.1, 0.2, 0.4]
NOISE_AMPS = [0.1, 0.2, 0.3]
APPLY_NOISE_TO_ATTRIBUTES = ["attributes"]
APPLY_NOISE_TO_CLASSES = ["classes"]
APPLY_BOTH = ["attributes","classes"]

all_results = []

# ---------
# BASELINE
# ---------
baseline = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_BOTH,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="baseline"
)
all_results.append(baseline)

# ---------
# PCA
# ---------

pca = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="pca",
    transform_fn=transform_pca
)
all_results.append(pca)

# ---------
# Z-SCORE
# ---------
zscore = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="zscore",
    transform_fn=remove_outliers_zscore
)
all_results.append(zscore)

# ---------
# DBSCAN
# ---------
dbscan = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="dbscan",
    transform_fn=remove_outliers_db
)
all_results.append(dbscan)

# ---------
# ISOLATION FOREST
# ---------
isf = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="isolation_forest",
    transform_fn=remove_outliers_isf
)
all_results.append(isf)

# ---------
# REGRESSION CLEANING
# ---------
regression = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="regression_cleaning",
    transform_fn=regression_reduce_noise
)
all_results.append(regression)

# ---------
# MEAN BINNING
# ---------
mean_bin = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="mean_binning",
    transform_fn=bin_attributes_mean
)
all_results.append(mean_bin)

# ---------
# MEDIAN BINNING
# ---------
median_bin = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_ATTRIBUTES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="median_binning",
    transform_fn=bin_attributes_median
)
all_results.append(median_bin)

# -------------------------
# LABEL NOISE REMOVAL (3x)
# -------------------------
label_cvcf= run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_CLASSES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="label_cvcf",
    transform_fn=remove_label_noise_cross_validated_committees_filter
)
all_results.append(label_cvcf)

label_enf = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_CLASSES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="label_enf",
    transform_fn= remove_label_noise_ensemble_filter
)
all_results.append(label_enf)

label_ipf = run_experiment(
    X_train_resampled,
    y_train_resampled,
    noise_levels=NOISE_LEVELS,
    noise_amps=NOISE_AMPS,
    apply_to=APPLY_NOISE_TO_CLASSES,
    models=[rfc, knc, gbc, svc, mlp],
    difficulties=indices,
    method_name="label_ipf",
    transform_fn=remove_label_noise_iterative_partitioning_filter
)
all_results.append(label_ipf)


In [ ]:
# =========================
# MERGE & COMPARE
# =========================
results = pd.concat(all_results, ignore_index=True)

comparison = compare_to(results, "baseline")

# =========================
# SAVE
# =========================
results.to_csv("denoising_all_results.csv", index=False)
comparison.to_csv("denoising_comparison_vs_baseline.csv", index=False)

print("DONE")
print("Results:", results.shape)
print("Comparison:", comparison.shape)

# Analiza rezultata

In [ ]:
comparison_sorted = comparison.sort_values(by=["f1_delta", "precision_delta"], ascending=[False, False])

In [ ]:
def filter_out(df, noise_level, noise_amp, noise_type, difficulty, classifier = None, method = None, select_top = 2):
    conditions = []
    if noise_level is None:
        noise_level = "'NA'"

    if noise_level is not None:
        conditions.append(f"noise_level == {noise_level}")
    if noise_amp is not None:
        conditions.append(f"noise_amp == {noise_amp}")
    if noise_type is not None:
        conditions.append(f"noise_type == '{noise_type}'")
    if method is not None:
        conditions.append(f"method == '{method}'")
    if difficulty is not None:
        conditions.append(f"difficulty == {difficulty}")
    if classifier is not None:
        conditions.append(f"classifier == '{classifier}'")
    
    condition = " and ".join(conditions)
    selected_cols = ['method','noise_level','noise_amp','noise_type','difficulty','classifier','f1','precision','f1_delta','precision_delta']

    partial = df.query(condition)[selected_cols]
    final = partial.sort_values(by=["f1", "precision", "f1_delta", "precision_delta"], ascending=[False, False, False, False])

    if select_top is None:
        return final
    return final[:select_top]

In [ ]:
def display_results(comparison, noise_level, noise_type, show_all_methods=False, top_k=5):
    classifiers = comparison['classifier'].unique()

    for noise_amp in [0.1, 0.2, 0.3]:
        for difficulty in [0, 1, 2]:
            print(f"\nŠum: {noise_level * 100 if noise_level else 0}% instanci, amp={noise_amp*100}%, {noise_type}, težina={difficulty}")
            
            for classifier in classifiers:
                baseline = filter_out(comparison, noise_level, noise_amp, noise_type, difficulty, 
                                    classifier=classifier, method='baseline', select_top=1)
                
                if len(baseline) == 0: continue
                baseline_row = baseline.iloc[0]
                
                if show_all_methods:
                    # Prikaži sve metode (ili top_k)
                    all_methods = filter_out(comparison, noise_level, noise_amp, noise_type, difficulty, 
                                           classifier=classifier, select_top=top_k)
                    
                    print(f"  {classifier}:")
                    for i, (idx, row) in enumerate(all_methods.iterrows(), 1):
                        marker = " [BASELINE]" if row['method'] == 'baseline' else ""
                        print(f"    {i}. {row['method']}{marker}: f1={row['f1']:.3f}, "
                              f"precision={row['precision']:.3f}, Δf1={row['f1_delta']:.3f}")
                else:
                    # Prikaži samo baseline vs najbolja metoda
                    all_methods = filter_out(comparison, noise_level, noise_amp, noise_type, difficulty, 
                                           classifier=classifier, select_top=100)
                    
                    non_baseline = all_methods[all_methods['method'] != 'baseline']
                    
                    if len(non_baseline) > 0:
                        top_row = non_baseline.iloc[0]
                        print(f"  {classifier}: baseline(f1={baseline_row['f1']:.3f}) vs "
                              f"{top_row['method']}(f1={top_row['f1']:.3f}, Δ={top_row['f1_delta']:.3f})")
                    else:
                        print(f"  {classifier}: baseline(f1={baseline_row['f1']:.3f}) vs -")
        if noise_level is None: break


## Šum među atributima

U slučaju kada nema dodatog šuma:
- Kod instanci težine 0 (najlakše) 
    - Sve metode nisu uticale na performanse ili su pogoršale dosta. Jedina metoda koja ne pogoršava previše, a u nekim slučajevima poboljšava performanse je PCA.
    - Najbolje rezultate daje GradientBoosting i SVC
- Kod instanci težine 1 (srednje) 
    - Opet najbolje rezultate daje GradientBoostingClassifier, pri čemu sve metode pozitivno utiču na performanse, međutim uklanjanje outlier-a ZScore metodom daje najbolje rezultate - skoro 30% poboljšanja
- Kod instanci težine 2 (najteže) 
    - Sve metode nisu uticale na performanse ili su pogoršale dosta. Jedina metoda koja ne pogoršava previše, a u nekim slučajevima poboljšava performanse je PCA.
    - Najbolje rezultate daje GradientBoosting i RandomForest

Ono što do sada možemo da zaključimo je da PCA najkonzistentnije daje bolje rezultate, pri čemu su najbolji modeli GradientBoosting, RandomForest i SVC

In [ ]:
display_results(comparison, None, "attributes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 10% instanci

- Sa amplitudom šuma od 10% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Nijedna metoda ne donosi poboljšanja, a neke i pogoršavaju.
    - Kod instanci težine 1 (srednje) 
        - Opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - skoro 30% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - Poboljšanja su bila blaga - PCA je pomogao kod SVCa, DBSCAN kod MLPa a regresija kod RandomForest klasifikatora.
        - Sva poboljšanja su bila blaga, dok su kod nekih modela metode kao što su DBSCAN i ZScore dosta pogoršali performanse

Ono što do sada možemo da zaključimo je da je PCA opet dobar izbor, GradientBoostingClassifier i dalje daje najbolje rezultate.
Redukovanje šuma uklanjanjem outliera (ZScore, DBSCAN, ISF) najbolje rade za instance težine 1.

- Sa amplitudom šuma od 20% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Nijedna metoda ne donosi poboljšanja, a neke i pogoršavaju.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - skoro 30% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - Poboljšanja su bila blaga - PCA je pomogao kod SVCa, regeresija kod MLPa i RandomForest klasifikatora.
        - Sva poboljšanja su bila blaga, dok su kod nekih modela metode kao što su DBSCAN i ZScore dosta pogoršali performanse

- Sa amplitudom šuma od 30% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Nijedna metoda ne donosi poboljšanja, a neke i pogoršavaju.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - skoro 30% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - Poboljšanja su bila blaga - PCA je pomogao kod SVCa, regeresija kod MLPa i RandomForest klasifikatora.
        - Sva poboljšanja su bila blaga, dok su kod nekih modela metode kao što su DBSCAN i ZScore dosta pogoršali performanse

Do sada je zaključak da:
- Najbolje rezultate konzistentno daje GradientBoosting klasifikator, praćen RandomForest, KNeighbors i SVC klasifikatorima. MLP daje najgore rezultate
- PCA nekad pogoršava performanse (jedva primetno), nekada blago poboljšava dok kod instance težine 1 postoje outlieri i njihovim uklanjanjem možemo dosta da poboljšamo rezultate

In [ ]:
display_results(comparison, 0.1, "attributes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 20% instanci

- Sa amplitudom šuma od 10% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Opet najbolje rezultate daje GradientBoostingClassifier. Otklanjanje outliera u ovoj grupi se opet pokazalo najboljim.
    - Kod instanci težine 2 (najteže) 
        - PCA je kod nekih metoda poboljšao blago performanse dok je kod GradientBoosting-a pogoršao

Sa povećanjem šuma PCA postaje gori izbor.

- Sa amplitudom šuma od 20% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - oko 25% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - PCA je kod nekih metoda poboljšao blago performanse dok je kod GradientBoosting-a pogoršao

- Sa amplitudom šuma od 30% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - oko 26% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - Poboljšanja su bila blaga, dok je PCA pogoršao rezultate GradientBoosting-a

In [ ]:
display_results(comparison, 0.2, "attributes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 40% instanci

- Sa amplitudom šuma od 10% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Opet najbolje rezultate daje GradientBoostingClassifier. Otklanjanje outliera u ovoj grupi se opet pokazalo najboljim.
    - Kod instanci težine 2 (najteže) 
        - PCA je kod nekih metoda poboljšao blago performanse dok je kod GradientBoosting-a pogoršao. Takođe regresija pomaže u nekim slučajevima

- Sa amplitudom šuma od 20% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - oko 30% poboljšanja
    - Kod instanci težine 2 (najteže) 
        - PCA je kod nekih metoda poboljšao blago performanse dok je kod GradientBoosting-a pogoršao. Regresija je blago pomogla kod RandomForest klasifikatora

- Sa amplitudom šuma od 30% početne vrednosti
    - Kod instanci težine 0 (najlakše) 
        - Standardno, instance težine 0 su dosta lake za predviđanje, tako da ni jedna metoda ne poboljšava rezultate.
    - Kod instanci težine 1 (srednje) 
        - Isto kao i kod 10% amplitude - opet najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a ZScore metodom daje najbolje rezultate - oko 27% poboljšanja
        - PCA i binning su blago pomogli kod RandomForest klasifikatora
    - Kod instanci težine 2 (najteže) 
        - Poboljšanja su bila blaga, dok je PCA pogoršao rezultate GradientBoosting-a

In [ ]:
display_results(comparison, 0.4, "attributes", show_all_methods=True, top_k=None)

<div style="background-color:#1212AA; height:auto; border-radius:10px; padding:16px; width:600px; color:white">
<h3>Zaključci</h3>
<ul>
<li>MLP daje najgore performanse, dok najbolje konzistentno daje GradientBoosting klasifikator, pogotovo kod instanci težine 1. Nakon toga slede SVC, RandomForest i KNeighbors.</li>
<li>
Najbolje metode za otklanjanje šuma u ovom slučaju su bili:
<ul>
    <li>PCA - blaga poboljšanja ali konzistentna - izuzev GradientBoosting klasifikatora kod težine 1</li>
    <li>Regression cleaning - blaga poboljšanja kod višeg nivoa šuma</li>
    <li>Mean i median binning - blaga poboljšanja u određenim slučajevima ali uglavnom pogoršavaju performanse</li>
    <li>DBSCAN, ZScore i ISF - najbolje performanse su dali kod instance težine 1, najverovatnije ova težina ima outliere. Uglavnom su DBSCAN i ZScore bili bolji.</li>
</ul>
</li>
<li>U konačnom modelu najbolje bi bilo 
<ul>
<li>Kombinovati GradientBoosting, SVC, RandomForest i KNeighbors klasifikatore</li>
<li>Primeniti PCA, regresiju i blaži DBSCAN ili ZScore sa širom varijansom</li>
</ul>
</ul>
</div>


## Šum među klasama

U slučaju bez dodatog šuma

- Kod instanci težine 0 (najlakše) 
    - Nijedna metoda ne donosi poboljšanja, a neke i pogoršavaju.
- Kod instanci težine 1 (srednje) 
    - Najbolje rezultate daje GradientBoostingClassifier, pri čemu uklanjanje outlier-a Ensemble filtering metodom daje najbolje rezultate - skoro 30% poboljšanja.
- Kod instanci težine 2 (najteže) 
    - U skoro svim slučajevima uklanjanje šuma pogoršava performanse, izuzev kod MLPa gde dovode do blagog poboljšanja.



In [ ]:
display_results(comparison, None, "classes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 10% instanci

- Kod instanci težine 0 (najlakše) 
    - Sve metode uglavnom poboljšavaju rezultate, pri čemu najbolje rezultate donosi Ensemble filter sa oko 5% pobolšanja.
- Kod instanci težine 1 (srednje) 
    - Kao i kod instanca težine 0, filtriranje poboljšava rezultate, pri čemu je Ensemble filter najbolji uglavnom (izuzev kod SVCa). Izuzetak je MLP gde filtriranje pogoršava rezultate.
- Kod instanci težine 2 (najteže) 
    - Kod težine 2, situacija se menja i sada se performanse pogoršavaju filtriranjem.

In [ ]:
display_results(comparison, 0.1, "classes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 20% instanci

- Kod instanci težine 0 (najlakše) 
    - Sve metode uglavnom poboljšavaju rezultate, pri čemu najbolje rezultate donosi Ensemble filter sa oko 16% pobolšanja.
- Kod instanci težine 1 (srednje) 
    - Kao i kod instanca težine 0, filtriranje poboljšava rezultate, pri čemu je Ensemble filter najbolji i to sa poboljšanjem od 20-30%. Izuzetak je MLP gde filtriranje pogoršava ili blago poboljšava ionako loše performanse.
- Kod instanci težine 2 (najteže) 
    - Kod težine 2, situacija se menja i sada se performanse pogoršavaju filtriranjem. Izuzetak je MLP gde Cross Validated Commitees filter blago poboljšava rezultate

In [ ]:
display_results(comparison, 0.2, "classes", show_all_methods=True, top_k=None)

U slučaju kada dodamo šum među 40% instanci

- Kod instanci težine 0 (najlakše) 
    - Sve metode uglavnom poboljšavaju rezultate, pri čemu najbolje rezultate donosi Ensemble filter sa oko 40-45% pobolšanja.
- Kod instanci težine 1 (srednje) 
    - Kao i kod instanca težine 0, filtriranje poboljšava rezultate, pri čemu je Ensemble filter najbolji i to sa poboljšanjem od 30-50%. Izuzetak je MLP gde se najbolje pakazao Iterative Partitioning filter.
- Kod instanci težine 2 (najteže) 
    - Kod težine 2, situacija se menja i sada se performanse uglavnom pogoršavaju filtriranjem. U nekim slučajevima Ensemble filter blago poboljšava rezultate

In [ ]:
display_results(comparison, 0.4, "classes", show_all_methods=True, top_k=None)

<div style="background-color:#1212AA; height:auto; border-radius:10px; padding:16px; width:600px; color:white">
<h3>Zaključci</h3>
<ul>
<li>MLP daje najgore performanse opet, dok najbolje konzistentno daje GradientBoosting i RandomForest klasifikatori</li>
<li>
Najbolje metoda za otklanjanje šuma u ovom slučaju je bila Ensemble filtering. U nekim slučajevima pogoršava performanse ali često je dovodila do poboljšanja, čak i do 50%.
</li>
<li>U konačnom modelu najbolje bi bilo 
<ul>
<li>Kombinovati GradientBoosting i RandomForest klasifikatore</li>
<li>Primeniti Ensemble filtering, pri čemu treba pojačati klasifikatore u ensemble-u, kao i povećati kriterijum za izbacivanje instance</li>
</ul>
</ul>
</div>


# Poslednje testiranje

Ako kombinujemo prethodne rezultate trebalo bi
- Kombinovati najbolje modele - GradientBoosting, RandomForest, KNeighbors i SVC i dodati im weights tim redom
- Iskoristiti prvo ZScore ili DBSCAN za otklanjanje outliera, pri čemu treba povećati kriterijum na osnovu kog se određuje da je instanca za izbacivanje.
- Zatim ukloniti šum medju klasama Ensemble filterom sa jačim klasifikatorima i većim kriterijumom
- Primeniti regresiju i blaži PCA

Zatim testirati model na čistom skupu, kao i kada se dodaju različiti stepeni šuma

In [ ]:

# =========================
# FINAL PIPELINE (implemented as requested in comments)
# =========================

rf = RandomForestClassifier(random_state=42)
gb = GradientBoostingClassifier(random_state=42)
knn = KNeighborsClassifier()
svc = SVC(probability=True, random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ("gb", gb),
        ("rf", rf),
        ("knn", knn),
        ("svc", svc),
    ],
    voting="soft",
    weights=[4, 3, 2, 1],  # ordered as requested
)

X_train, y_train = X_train_resampled.copy(), y_train_resampled.copy()
X_test, y_test = X_train_resampled.copy(), y_train_resampled.copy()

X_clean, y_clean, _ = remove_outliers_zscore(X_train, y_train, threshold=4.0)

X_clean, y_clean, _ = remove_label_noise_ensemble_filter(
    X_clean,
    y_clean,
    n_splits = 7,
    voting_threshold = 3 // 4,
)

# 3) Regression noise reduction + PCA(0.99)
X_reg, _ = regression_reduce_noise(X_clean, y_clean)
X_reg_pca, _ = transform_pca(X_reg, y_clean, n_comp=0.99)

# Apply same transforms to test
X_test_reg, _ = regression_reduce_noise(X_test, y_test)
X_test_reg_pca, _ = transform_pca(X_test_reg, y_test, n_comp=0.99)


In [ ]:
pairs = generate_test_pairs(X_reg_pca, y_clean, [ensemble], "Final", indices, X_test_reg_pca, y_test)

df = TestClassifiers(pairs, None, None, "", method = "Final")

df